In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
 
pd.set_option("future.no_silent_downcasting", True)
 
from laparoscopy_helpers.data_cleaning import to_snake_case, clean_surgical_df
from poor_patient_helpers.data_cleaning import load_all_endo, load_all_mercy, load_all_safe, build_final

In [2]:
path_corrected_names = '../Nkhoma_data/poor_patients_funds_data_corrected_names'

In [3]:
os.listdir(path_corrected_names)

['combined_clean_corrected.xlsx',
 'combined_clean.xlsx',
 'combined_clean_renamed.xlsx',
 'not_in_TB_colored.xlsx']

In [4]:
import pandas as pd

df1 = pd.read_excel(f"{path_corrected_names}/combined_clean.xlsx")
df2 = pd.read_excel(f"{path_corrected_names}/combined_clean_renamed.xlsx")

In [5]:
df2 = df2.sort_values('theatre_book_index')
df2

,theatre_book_index,hospital_id,date_of_surgery,first_name,last_name,age_years,sex,village,surgeon,1st_assistent_instructor,...,surgery_severity,asascore,year_of_birth,operation_time_minutes,starting_time,continous_tb,year,Unnamed: 32,Unnamed: 33,Unnamed: 34
0,220001,NaN,2022-01-01,ELIFA,SUMATI,26.0,F,Nkhonde,Obs/Gyn,NaN,...,NaN,NaN,1997,NaN,NaN,NaN,2022.0,NaN,NaN,NaN
1,220002,NaN,2022-01-01,SIYATU,ISAAC,27.0,F,Mozambique,Obs/Gyn,NaN,...,NaN,NaN,1996,NaN,NaN,NaN,2022.0,NaN,NaN,NaN
2,220003,NaN,2022-01-02,LONESS,MAPEMPHERO,25.0,F,Chembe,Obs/Gyn,NaN,...,NaN,NaN,1998,NaN,NaN,NaN,2022.0,NaN,NaN,NaN
3,220004,NaN,2022-01-03,SAIZI,NEDSON,48.0,M,Chilikumanda,Limbe,Caleb,...,Major,ASA 3,1975,NaN,NaN,NaN,2022.0,NaN,NaN,NaN
4,220005,NaN,2022-01-03,BEATRICE,HEZEKIA,26.0,F,Mazengera,Obs/Gyn,NaN,...,NaN,NaN,1997,NaN,NaN,NaN,2022.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7401,251598,NaN,2025-12-27,MODAKIAH,SAIMON,23.0,M,Mazengera,Limbe,Mallen,...,Major,ASA 2,NaN,NaN,NaN,250611.0,2025.0,NaN,NaN,NaN
7402,251599,NaN,2025-12-27,CATHERINE,CHIPOTE,43.0,F,Chitekwere,Limbe,Mallen,...,Major,ASA 4,NaN,NaN,NaN,250631.0,2025.0,NaN,NaN,NaN
7403,251600,NaN,2025-12-28,CATHERINE,CHIPOTE,43.0,F,Chitekwere,Limbe,Jonathan,...,Major,ASA 2,NaN,NaN,NaN,250632.0,2025.0,NaN,NaN,NaN
7404,251601,NaN,2025-12-29,MARVELLOUS,SAMUEL,45.0,F,Mazengera,Caleb,Limbe,...,Minor,ASA 2,NaN,NaN,NaN,250503.0,2025.0,NaN,NaN,NaN


In [6]:
new_rows = df2[~df2['theatre_book_index'].isin(df1['theatre_book_index'])]
# Append them to df1
df1 = pd.concat([df1, new_rows], ignore_index=True)
df1 = df1.sort_values('theatre_book_index')

In [7]:
# Merge on theatre_book_index to align rows
comparison = df1.merge(df2[['theatre_book_index', 'first_name', 'last_name']], 
                        on='theatre_book_index', 
                        suffixes=('_df1', '_df2'))

# Filter only rows where something differs
diffs = comparison[
    (comparison['first_name_df1'] != comparison['first_name_df2']) |
    (comparison['last_name_df1'] != comparison['last_name_df2'])
][['theatre_book_index', 'first_name_df1', 'first_name_df2', 'last_name_df1', 'last_name_df2']]

print(f"{len(diffs)} rows differ")
print(diffs)

1821 rows differ
      theatre_book_index first_name_df1 first_name_df2 last_name_df1  \
77                220078            NaN            NaN           NaN   
144               220145            NaN            NaN           NaN   
217               220218            NaN            NaN           NaN   
382               220383           LUMA          LUNIA        TALIFA   
554               220555         WONIZA       WELUZANI          BISE   
...                  ...            ...            ...           ...   
7405              251594          FRANK     MARVELLOUS    MTAMBARIKA   
7407              251596          JAMES  JAMES HENRY/          HENRY   
7408              251597        MODECAI       MODAKIAH         SYMON   
7409              251598        MODECAI       MODAKIAH         SYMON   
7412              251601      CATHERINE     MARVELLOUS        SAMUEL   

     last_name_df2  
77             NaN  
144            NaN  
217            NaN  
382         JALAFI  
554          

In [8]:
# Exclude rows where both are NaN (not real differences)
real_diffs = diffs[
    ~(
        (comparison['first_name_df1'].isna() & comparison['first_name_df2'].isna()) |
        (comparison['last_name_df1'].isna() & comparison['last_name_df2'].isna())
    )
]

# Actually, cleaner: only keep rows where values differ AND at least one is not NaN
real_diffs = diffs[
    (comparison['first_name_df1'].fillna('') != comparison['first_name_df2'].fillna('')) |
    (comparison['last_name_df1'].fillna('') != comparison['last_name_df2'].fillna(''))
]

print(f"{len(real_diffs)} real differences")
print(real_diffs)

1697 real differences
      theatre_book_index first_name_df1 first_name_df2 last_name_df1  \
382               220383           LUMA          LUNIA        TALIFA   
554               220555         WONIZA       WELUZANI          BISE   
645               220646       LAWRENCE       LAWRENCE        KANTHU   
842               220843         HASSAN        HUSSEIN         SAIDI   
845               220846         NIMROD        NIMRODI        PARASU   
...                  ...            ...            ...           ...   
7405              251594          FRANK     MARVELLOUS    MTAMBARIKA   
7407              251596          JAMES  JAMES HENRY/          HENRY   
7408              251597        MODECAI       MODAKIAH         SYMON   
7409              251598        MODECAI       MODAKIAH         SYMON   
7412              251601      CATHERINE     MARVELLOUS        SAMUEL   

       last_name_df2  
382           JALAFI  
554             BISE  
645   KUTHAKWAWANTHU  
842            SAIDI 

/tmp/ipykernel_6593/1311615606.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  real_diffs = diffs[
/tmp/ipykernel_6593/1311615606.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  real_diffs = diffs[


In [9]:
df1 = df1.merge(df2[['theatre_book_index', 'first_name', 'last_name']], 
                on='theatre_book_index', 
                suffixes=('_old', ''))
df1.drop(columns=['first_name_old', 'last_name_old'], inplace=True)

In [10]:
df1

,theatre_book_index,hospital_id,date_of_surgery,age_years,sex,village,surgeon,1st_assistent_instructor,2nd_assistent,anaestesist,...,year_of_birth,operation_time_minutes,starting_time,continous_tb,year,Unnamed: 32,Unnamed: 33,Unnamed: 34,first_name,last_name
0,220001,NaN,2022-01-01,26.0,F,Nkhonde,Obs/Gyn,NaN,NaN,Frank Nkhoma,...,1997.0,NaN,NaN,NaN,2022.0,NaN,NaN,NaN,ELIFA,SUMATI
1,220002,NaN,2022-01-01,27.0,F,Mozambique,Obs/Gyn,NaN,NaN,Frank Nkhoma,...,1996.0,NaN,NaN,NaN,2022.0,NaN,NaN,NaN,SIYATU,ISAAC
2,220003,NaN,2022-01-02,25.0,F,Chembe,Obs/Gyn,NaN,NaN,Frank Nkhoma,...,1998.0,NaN,NaN,NaN,2022.0,NaN,NaN,NaN,LONESS,MAPEMPHERO
3,220004,NaN,2022-01-03,48.0,M,Chilikumanda,Limbe,Caleb,NaN,Frank Nkhoma,...,1975.0,NaN,NaN,NaN,2022.0,NaN,NaN,NaN,SAIZI,NEDSON
4,220005,NaN,2022-01-03,26.0,F,Mazengera,Obs/Gyn,NaN,NaN,Frank Nkhoma,...,1997.0,NaN,NaN,NaN,2022.0,NaN,NaN,NaN,BEATRICE,HEZEKIA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7409,251598,NaN,2025-12-27,23.0,M,Mazengera,Limbe,Mallen,NaN,George Mponda,...,NaN,NaN,NaN,250611.0,2025.0,NaN,NaN,NaN,MODAKIAH,SAIMON
7410,251599,NaN,2025-12-27,43.0,F,Chitekwere,Limbe,Mallen,NaN,George Mponda,...,NaN,NaN,NaN,250631.0,2025.0,NaN,NaN,NaN,CATHERINE,CHIPOTE
7411,251600,NaN,2025-12-28,43.0,F,Chitekwere,Limbe,Jonathan,NaN,George Mponda,...,NaN,NaN,NaN,250632.0,2025.0,NaN,NaN,NaN,CATHERINE,CHIPOTE
7412,251601,NaN,2025-12-29,45.0,F,Mazengera,Caleb,Limbe,NaN,NaN,...,NaN,NaN,NaN,250503.0,2025.0,NaN,NaN,NaN,MARVELLOUS,SAMUEL


In [11]:
# Drop unnamed columns
df1 = df1.drop(columns=['Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34'])

no_name_rows = df1[df1['first_name'].isna() | df1['last_name'].isna()]
#df1 = df1.dropna(subset=['first_name', 'last_name'])

print(f"Kept: {len(df1)}, Dropped: {len(no_name_rows)}")
print(df1.shape)

Kept: 7414, Dropped: 128
(7414, 32)


In [13]:
df1
df1.to_excel(f"../Nkhoma_data/poor_patients_funds_data_corrected_names/combined_clean_corrected.xlsx", index=False, engine="openpyxl")